# 🧠 Day 2: Handwritten Digit Classification with an ANN (Keras)

**Goal:** Teach a computer to look at an image of a handwritten digit (0-9) and tell us which digit it is.

### What you will learn
1. How a computer "sees" an image (spoiler: just numbers!)
2. How to prepare image data for a neural network
3. How to build an **Artificial Neural Network (ANN)** with Keras
4. What *epoch*, *batch size*, *loss*, *optimizer* and *activation function* mean
5. How to train, evaluate and test the model, even on your **own handwriting**

### How to use this notebook
- Run cells **one by one** with `Shift + Enter`.
- Read the explanation above each code cell *before* running it.
- Look for the **🧪 Try it yourself** boxes. Change numbers and see what happens!

> **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for faster training (optional, this model also runs fine on CPU).

---
## 📦 Step 1: Import the libraries

Before we start, we bring in the tools we need:

| Library | What it does |
|---|---|
| `numpy` | Fast math on arrays (images are arrays of numbers) |
| `matplotlib` | Draw graphs and show images |
| `seaborn` | Pretty heatmaps (for the confusion matrix) |
| `tensorflow` / `keras` | Build and train the neural network |
| `sklearn` | Evaluation tools (confusion matrix, report) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

---
## 📥 Step 2: Load the MNIST dataset

**MNIST** is the "Hello World" of deep learning. It contains **70,000 images** of handwritten digits collected from students and census workers.

- **60,000** images for **training** (the model learns from these)
- **10,000** images for **testing** (the model has never seen these, so we use them to check how well it really learned)

Keras can download it for us in one line. The data comes in 4 parts:

| Variable | Meaning |
|---|---|
| `x_train` | Training **images** |
| `y_train` | Training **labels** (the correct digit for each image) |
| `x_test` | Testing images |
| `y_test` | Testing labels |

> 💡 Remember Day 1? Same idea: **x = features**, **y = label**. This is *supervised learning* again.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape: ", x_test.shape)
print("y_test shape: ", y_test.shape)

### 🔍 Understanding the shapes

- `x_train.shape = (60000, 28, 28)` means **60,000 images**, each **28 pixels tall × 28 pixels wide**.
- `y_train.shape = (60000,)` means **60,000 labels**, one number (0-9) per image.

Each image is tiny: only 28 × 28 = **784 pixels**.

---
## 👀 Step 3: Look at the data

Always look at your data first. Let's display the first 10 training images with their labels.

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap="gray")   # cmap="gray" -> show in black & white
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("First 10 training images", y=1.05)
plt.show()

### 🤔 How does a computer "see" an image?

A computer does **not** see a picture. It sees a **grid of numbers**.

Each pixel is a number from **0 (black)** to **255 (white)**. Let's print the actual numbers of the first image.

In [ ]:
# Print the raw pixel values of the first image, as a 28x28 grid of numbers
np.set_printoptions(linewidth=250)
print("This image is labelled:", y_train[0])
print(x_train[0])

Notice: lots of `0`s (black background) and bigger numbers (up to `255`) where the pen touched the paper. **This grid of numbers is what we feed into the neural network.**

Let's also see the numbers laid over the image:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(x_train[0], cmap="gray")
for row in range(28):
    for col in range(28):
        value = x_train[0][row][col]
        ax.text(col, row, str(value), ha="center", va="center", fontsize=5,
                color="white" if value < 128 else "black")
ax.set_title(f"Pixel values for the digit {y_train[0]}")
plt.show()

### 📊 Quick check: is the data balanced?

If 90% of images were the digit "1", the model could cheat by always guessing "1". Let's count how many images exist for each digit.

In [ ]:
digits, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(digits, counts, color="steelblue")
plt.xticks(digits)
plt.xlabel("Digit")
plt.ylabel("Number of training images")
plt.title("Class distribution in the training set")
plt.show()

The classes are roughly balanced (about 6,000 each), so the data is fair. ✅

---
## 🧹 Step 4: Preprocess the data

Neural networks learn best when inputs are **small and consistent**. We do two things:

### 4a. Normalize the pixels (scale 0-255 → 0-1)
Dividing every pixel by 255 squeezes values into the range 0 to 1.

**Why?** Large numbers make learning unstable and slow. Small numbers make gradient descent (the learning process) smooth, like walking down a gentle slope instead of a cliff.

In [ ]:
x_train = x_train / 255.0
x_test  = x_test  / 255.0

print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())

### 4b. One-hot encode the labels

Our labels are numbers like `3` or `7`. But the network's output layer will have **10 neurons** (one per digit), each giving a probability. So we convert each label into a list of 10 numbers where only the correct position is `1`:

| Label | One-hot vector |
|---|---|
| 0 | `[1,0,0,0,0,0,0,0,0,0]` |
| 3 | `[0,0,0,1,0,0,0,0,0,0]` |
| 7 | `[0,0,0,0,0,0,0,1,0,0]` |

We keep the original labels in `y_test_labels` because we will need them later for the confusion matrix.

In [ ]:
y_train_labels = y_train.copy()   # keep the original numbers (0-9)
y_test_labels  = y_test.copy()

y_train = to_categorical(y_train, num_classes=10)
y_test  = to_categorical(y_test,  num_classes=10)

print("Original label:", y_train_labels[0])
print("One-hot label: ", y_train[0])

> ℹ️ **Note:** We don't reshape the images here. The first layer of our model (`Flatten`) will do it for us in the next step.

---
## 🏗️ Step 5: Build the ANN

Now the exciting part: designing the brain of our model.

### The architecture

```
 Input image        Flatten        Hidden layer 1     Hidden layer 2      Output layer
  (28 × 28)   →   (784 numbers) →  (128 neurons)  →   (64 neurons)   →   (10 neurons)
                                       ReLU              ReLU              Softmax
```

### Layer by layer

**1. `Flatten(input_shape=(28, 28))`**
A dense (fully connected) layer expects a **flat list** of numbers, not a 2D grid. Flatten unrolls the 28×28 grid into a list of **784** numbers.

**2. `Dense(128, activation="relu")`**
- `Dense` = every neuron is connected to **every** input from the previous layer.
- `128` = number of neurons. Each neuron looks at all 784 pixels and learns a different pattern.
- `relu` = **activation function**. It outputs `max(0, x)`: negative values become 0, positive values pass through. It lets the network learn **non-linear** patterns. Without an activation function, the whole network would behave like a single straight line!

**3. `Dense(64, activation="relu")`**
A second hidden layer that combines the patterns from the first layer into higher-level ones.

**4. `Dense(10, activation="softmax")`**
- **10 neurons** = one for each digit (0-9).
- `softmax` converts the 10 raw scores into **probabilities that add up to 1** (for example 0.02, 0.01, 0.95, ...). The highest probability is the model's prediction.

In [ ]:
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),        # 28x28 grid -> 784 numbers
    layers.Dense(128, activation="relu"),         # hidden layer 1
    layers.Dense(64, activation="relu"),          # hidden layer 2
    layers.Dense(10, activation="softmax")        # output layer: 10 digit probabilities
])

model.summary()

### 🧮 Understanding `model.summary()`

The **Param #** column counts the numbers the network must learn (weights and biases).

For a Dense layer: **parameters = (inputs × neurons) + neurons (biases)**

| Layer | Calculation | Parameters |
|---|---|---|
| Dense 1 | 784 × 128 + 128 | **100,480** |
| Dense 2 | 128 × 64 + 64 | **8,256** |
| Dense 3 (output) | 64 × 10 + 10 | **650** |
| **Total** | | **109,386** |

So our little network has about **109 thousand** knobs that training will tune. (ChatGPT-sized models have *hundreds of billions*!)

- **Weights**: how important each input is.
- **Biases**: a shift that lets the neuron adjust its output.

---
## ⚙️ Step 6: Compile the model

Before training, we must tell the model **three things**:

| Setting | Our choice | Meaning |
|---|---|---|
| **Optimizer** | `adam` | The algorithm that *updates the weights* to reduce error. Think of it as a smart hiker walking downhill in fog toward the lowest point. Adam is a popular, reliable choice. |
| **Loss function** | `categorical_crossentropy` | Measures **how wrong** the predictions are. High loss = bad predictions. Training tries to make this as small as possible. We use this one because we have one-hot labels and multiple classes. |
| **Metric** | `accuracy` | What *we* want to watch: the % of images classified correctly. |

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled and ready to train!")

---
## 🏋️ Step 7: Train the model

Now the model learns from the data by calling `model.fit()`.

### Key terms

- **Epoch**: one complete pass through the **entire** training set. `epochs=10` means the model sees all 60,000 images 10 times.
- **Batch size**: instead of learning from all images at once, the model learns from small groups. `batch_size=128` means the weights are updated after every 128 images. Number of updates per epoch = 54,000 ÷ 128 ≈ 422.
- **Validation split**: `validation_split=0.1` sets aside **10%** of training data (6,000 images). The model does *not* learn from them. We use them after each epoch to check for **overfitting**.

### What happens inside one training step?
1. **Forward pass**: images go through the network and produce predictions.
2. **Loss calculation**: compare predictions with the true labels.
3. **Backpropagation**: calculate how much each weight contributed to the error.
4. **Gradient descent (Adam)**: nudge each weight slightly to reduce the error.

Repeat thousands of times, and the network gets smarter!

▶️ Run the cell and watch `accuracy` go **up** and `loss` go **down**.

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

### 📖 Reading the output

Each epoch prints four numbers:

| Name | Meaning |
|---|---|
| `loss` | Error on the **training** data (lower is better) |
| `accuracy` | Accuracy on the **training** data |
| `val_loss` | Error on the **validation** data (unseen during learning) |
| `val_accuracy` | Accuracy on the **validation** data |

---
## 📈 Step 8: Plot the learning curves

A graph tells the story of training much better than numbers. We plot accuracy and loss for training and validation across epochs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy plot
axes[0].plot(history.history["accuracy"], label="Training accuracy", marker="o")
axes[0].plot(history.history["val_accuracy"], label="Validation accuracy", marker="o")
axes[0].set_title("Accuracy over epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history["loss"], label="Training loss", marker="o")
axes[1].plot(history.history["val_loss"], label="Validation loss", marker="o")
axes[1].set_title("Loss over epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.show()

### 🧐 How to read these graphs

- ✅ **Good learning:** accuracy goes up, loss goes down, and the training and validation lines stay **close together**.
- ⚠️ **Overfitting:** training accuracy keeps rising but validation accuracy stops improving or drops (and validation loss goes *up*). The model is **memorizing** instead of **understanding**. (Remember the Day 1 analogy of a student who memorizes answers?)
- ⚠️ **Underfitting:** both accuracies stay low. The model is too simple or hasn't trained enough.

**Question for the class:** Do you see any overfitting in your graph?

---
## 🎯 Step 9: Evaluate on the test set

The validation set was used *during* training. The **test set** is the real final exam: images the model has **never** seen. This tells us how it will perform in the real world.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

🎉 A simple ANN typically scores around **97-98%**. That means out of 10,000 unseen digits, it gets roughly 9,700+ right!

---
## 🔮 Step 10: Make predictions

`model.predict()` returns, for every image, **10 probabilities** (one per digit). The digit with the highest probability is the prediction. We find it with `np.argmax()`.

In [ ]:
predictions = model.predict(x_test)          # shape: (10000, 10)
predicted_labels = np.argmax(predictions, axis=1)   # pick the highest-probability digit

print("Shape of predictions:", predictions.shape)
print("\nProbabilities for the first test image:")
print(np.round(predictions[0], 3))
print("\nPredicted digit:", predicted_labels[0])
print("Actual digit:   ", y_test_labels[0])

### 👁️ Visualize predictions for a few test images

- 🟢 **Green title** = correct prediction
- 🔴 **Red title** = wrong prediction

The percentage is the model's **confidence**.

In [ ]:
plt.figure(figsize=(14, 6))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    pred = predicted_labels[i]
    actual = y_test_labels[i]
    confidence = predictions[i][pred] * 100
    color = "green" if pred == actual else "red"
    plt.title(f"Pred: {pred} ({confidence:.0f}%)\nActual: {actual}", color=color, fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

### 📊 What does the model "think"? Probability bars

Let's pick **one image** and see the probability the model gave to each digit. Change `index` to explore different images.

In [ ]:
index = 7    # 🧪 Try changing this number (0 to 9999)

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.imshow(x_test[index], cmap="gray")
plt.title(f"Actual digit: {y_test_labels[index]}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(range(10), predictions[index], color="orange")
plt.xticks(range(10))
plt.xlabel("Digit")
plt.ylabel("Probability")
plt.title(f"Model prediction: {predicted_labels[index]}")

plt.tight_layout()
plt.show()

---
## 🧩 Step 11: Confusion matrix & classification report

Accuracy gives one number, but it doesn't tell us **which digits get confused**. The **confusion matrix** does:

- **Rows** = actual digit
- **Columns** = predicted digit
- The **diagonal** = correct predictions (should be bright!)
- Off-diagonal cells = mistakes (e.g. row 4, column 9 means "actual 4, predicted 9")

In [ ]:
cm = confusion_matrix(y_test_labels, predicted_labels)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted digit")
plt.ylabel("Actual digit")
plt.title("Confusion Matrix")
plt.show()

**Discussion:** Which pairs of digits does the model confuse most? (Often 4↔9, 3↔5, 7↔2. They look similar even to humans!)

Now the **classification report**, with the Day 1 metrics again:
- **Precision**: of all images the model called "7", how many really were 7?
- **Recall**: of all real 7s, how many did the model find?
- **F1-score**: a balance of the two.

In [ ]:
print(classification_report(y_test_labels, predicted_labels))

---
## ❌ Step 12: Look at the mistakes

The best way to improve a model is to understand its errors. Let's find the images it got wrong and view them.

In [ ]:
wrong_idx = np.where(predicted_labels != y_test_labels)[0]
print(f"Total misclassified images: {len(wrong_idx)} out of {len(y_test_labels)}")

plt.figure(figsize=(14, 6))
for i, idx in enumerate(wrong_idx[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(f"Pred: {predicted_labels[idx]}\nActual: {y_test_labels[idx]}", color="red", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

**Discussion:** Would *you* have gotten these right? Some are so messy that even humans struggle. That's the nature of real-world data.

---
## ✍️ Step 13: Test it on YOUR handwriting!

Now the fun part. 

### Instructions
1. Write a single digit (0-9) with a **thick dark pen on white paper** (or draw it in Paint or a phone drawing app).
2. Take a photo or screenshot and **crop it close** around the digit so it is roughly square.
3. Run the cell below and upload the image.

The code will:
- convert it to grayscale,
- resize it to **28×28**,
- **invert the colors if needed** (MNIST digits are *white on black*, but paper is *black on white*),
- normalize it and ask the model to predict.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()   # a file picker will appear

for filename in uploaded.keys():
    # 1. Open image and convert to grayscale
    img = Image.open(filename).convert("L")

    # 2. Resize to 28x28 pixels
    img = img.resize((28, 28))
    arr = np.array(img)

    # 3. MNIST has white digits on a black background. If our image is mostly bright, invert it.
    if arr.mean() > 127:
        arr = 255 - arr

    # 4. Normalize to 0-1 and add a batch dimension -> shape (1, 28, 28)
    arr = arr / 255.0
    arr_input = arr.reshape(1, 28, 28)

    # 5. Predict
    probs = model.predict(arr_input, verbose=0)[0]
    pred = np.argmax(probs)

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(arr, cmap="gray")
    plt.title("What the model sees (28x28)")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.bar(range(10), probs, color="purple")
    plt.xticks(range(10))
    plt.title(f"Prediction: {pred} ({probs[pred]*100:.1f}% sure)")
    plt.tight_layout()
    plt.show()

> 💡 **If the prediction is wrong:** that's normal! Our model was trained on clean, centered digits. Try writing thicker, centered and bigger. This shows an important lesson: **models only perform well on data similar to what they were trained on.**

---
## 🧪 Step 14: Experiment time! (Hands-on challenge)

Real machine learning is about **experimenting**. Below is a helper function that builds an ANN with settings **you choose**, trains it, and reports the test accuracy.

Try different values and fill in the table at the bottom.

In [ ]:
def build_and_train(neurons=128, num_hidden_layers=2, activation="relu",
                    dropout_rate=0.0, epochs=5, learning_rate=0.001):
    """Builds, trains and evaluates an ANN with the chosen settings."""
    m = keras.Sequential()
    m.add(layers.Flatten(input_shape=(28, 28)))

    for _ in range(num_hidden_layers):
        m.add(layers.Dense(neurons, activation=activation))
        if dropout_rate > 0:
            m.add(layers.Dropout(dropout_rate))   # randomly switches off neurons to fight overfitting

    m.add(layers.Dense(10, activation="softmax"))

    m.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
              loss="categorical_crossentropy", metrics=["accuracy"])

    m.fit(x_train, y_train, epochs=epochs, batch_size=128, validation_split=0.1, verbose=0)
    _, acc = m.evaluate(x_test, y_test, verbose=0)
    print(f"neurons={neurons}, layers={num_hidden_layers}, activation={activation}, "
          f"dropout={dropout_rate}, epochs={epochs}, lr={learning_rate}  ->  Test accuracy: {acc*100:.2f}%")
    return acc

### 🧪 Experiment A: Does a bigger network help?

In [ ]:
for n in [8, 32, 128, 512]:
    build_and_train(neurons=n, num_hidden_layers=1)

### 🧪 Experiment B: Does depth (more layers) help?

In [ ]:
for layer_count in [0, 1, 2, 4]:
    build_and_train(neurons=128, num_hidden_layers=layer_count)

*(With `0` hidden layers the model is just a single layer, like logistic regression. Notice how much hidden layers help!)*

### 🧪 Experiment C: What if we remove the activation function?

In [ ]:
build_and_train(activation="relu")
build_and_train(activation="linear")   # "linear" = no activation function

Without a non-linear activation, the layers collapse into one big linear function and the network loses its power.

### 🧪 Experiment D: Learning rate (the size of each step)

In [ ]:
for lr in [0.00001, 0.001, 0.1]:
    build_and_train(learning_rate=lr)

- **Too small** → learning is painfully slow.
- **Just right** → smooth learning.
- **Too big** → the model jumps around and may never settle.

### 🧪 Experiment E: Dropout against overfitting

In [ ]:
build_and_train(neurons=512, num_hidden_layers=2, dropout_rate=0.0, epochs=10)
build_and_train(neurons=512, num_hidden_layers=2, dropout_rate=0.3, epochs=10)

### 📝 Your results table (fill in during the workshop)

| Experiment | Setting | Test accuracy | What I noticed |
|---|---|---|---|
| A | 8 neurons | | |
| A | 512 neurons | | |
| B | 0 hidden layers | | |
| B | 4 hidden layers | | |
| C | linear activation | | |
| D | lr = 0.1 | | |
| E | dropout 0.3 | | |

**Challenge:** Can you beat your classmates? Find the combination of settings with the **highest test accuracy**!

---
## 💾 Step 15: Save the model (optional)

Once trained, you can save a model and reload it later without retraining.

In [ ]:
model.save("digit_ann_model.keras")
print("Model saved!")

# Load it back
loaded_model = keras.models.load_model("digit_ann_model.keras")
loss, acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print(f"Reloaded model test accuracy: {acc*100:.2f}%")

---
## 🏁 Summary: what we did today

1. **Loaded** MNIST: 70,000 images of handwritten digits.
2. **Learned** that an image is just a grid of pixel numbers (0-255).
3. **Preprocessed**: normalized pixels and one-hot encoded labels.
4. **Built an ANN**: Flatten → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax).
5. **Compiled** with the Adam optimizer, cross-entropy loss and accuracy metric.
6. **Trained** for several epochs and plotted learning curves.
7. **Evaluated** on unseen data (~97-98% accuracy).
8. **Analyzed** mistakes with a confusion matrix.
9. **Tested** on our own handwriting.
10. **Experimented** with layers, neurons, activations, learning rate and dropout.

### 📚 Quick glossary

| Term | Simple meaning |
|---|---|
| **Neuron** | A tiny calculator: weighted sum of inputs + bias → activation |
| **Weight** | How much a neuron cares about an input |
| **Activation function** | Adds non-linearity (ReLU, Sigmoid, Softmax) |
| **Epoch** | One full pass through the training data |
| **Batch size** | Images processed before each weight update |
| **Loss** | How wrong the model is |
| **Optimizer** | The method that adjusts weights to reduce loss |
| **Backpropagation** | Working out how each weight contributed to the error |
| **Overfitting** | Memorizing training data, failing on new data |
| **Dropout** | Randomly turning off neurons during training to reduce overfitting |

### ❓ Think about it
1. Why do we divide pixel values by 255?
2. Why does the output layer have exactly 10 neurons?
3. What would happen if we trained for 100 epochs?
4. An ANN treats each pixel independently and ignores neighbouring pixels. Why might that be a problem? *(Hint: this is exactly what **CNNs** solve, our next topic! 🚀)*

### 🚀 Take-home challenges
- Repeat the project with **Fashion-MNIST** (`keras.datasets.fashion_mnist`). Same code, harder problem!
- Add `EarlyStopping` so the model stops when validation loss stops improving.
- Try to reach **98.5%+** test accuracy using only an ANN.

---
*Next class: Convolutional Neural Networks (CNNs), a network that actually understands images.*